# Suite ANY — Values of any schema

`Schemas.OfAny.Value` is the kind of a value of any schema: built with `schemas`, it is restricted to them, and with
none, it is unrestricted. A restricted value is written bare, its schema the one whose plain form it takes; an
unrestricted one is tagged with its schema, a basic scalar's tag or a registered schema's name,
and is assigned as a value, built standalone through the store where it is not a scalar.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import document, raises, root_of, single

store = B.Store()
Int, Text, Float, Bytes, Bool = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema(), store.Schemas.OfFloat.as_schema(), store.Schemas.OfBytes.as_schema(), store.Schemas.OfBool.as_schema()  # the store's own
with store.transaction():
    Phone = store.Schemas.OfObject(lambda o: o.name("Phone").properties([lambda t: t.name("number").schema([Text])]))
    Code = store.Schemas.OfUnion(lambda u: (
        u
        .name("Code")
        .branches([
            lambda t: t.name("n").schema([Int]),
            lambda t: t.name("s").schema([Text]),
        ])
    ))
    Stamp = store.Schemas.OfIntersection(lambda i: (
        i
        .name("Stamp")
        .parts([
            lambda t: t.name("at").schema([Int]),
            lambda t: t.name("by").schema([Text]),
        ])
    ))
    Tags = store.Schemas.OfContainer(lambda c: c.name("Tags").index([Int]).element([Text]))
    Labels = store.Schemas.OfContainer(lambda c: c.name("Labels").index([Text]).element([Text]))  # a dict by text: a mapping
    Cell = store.Schemas.OfObject(lambda o: o.name("Cell").properties([
        lambda t: t.name("r").schema([Int]),
        lambda t: t.name("c").schema([Int]),
    ]))
    Grid = store.Schemas.OfContainer(lambda c: c.index([Cell]).element([Int]))  # a tensor: nested arrays, or flat with its shape
    One = store.Schemas.OfAny(lambda a: a.schemas([Int, Text, Phone]))
    Free = store.Schemas.OfAny(lambda a: a.name("Free"))
    Card = store.Schemas.OfObject(lambda o: (
        o
        .name("Card")
        .ref(True)
        .properties([
            lambda t: t.name("one").schema([One]),
            lambda t: t.name("free").schema([Free]),
            lambda t: t.name("many").schema([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Free]))]),
            lambda t: t.name("tags").schema([store.Schemas.OfAny(lambda a: a.schemas([Bool, Tags]))]),
        ])
    ))


def held(prop):
    """What a property holds, read through its visitor: a scalar, or a visitor's schema's name; None where absent."""
    found = []
    prop.as_value(lambda a: a.as_scalar(lambda s: found.extend([] if s.get() is None else [s.get()]) or s)
                  .as_object(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_union(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_intersection(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_container(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v))
    return found[0] if found else None

## ANY-01 · An `OfAny` restricted to the schemas selected, or to none; its schemas told apart on the wire

In [ ]:
with store.transaction():
    assert type(One) is S.OfAny.Value and One.schemas == (Int, Text, Phone) and One.validate() == []
    assert Free.schemas == () and Free.validate() == [] and Card.validate() == []
    only = store.Schemas.OfAny(lambda a: a.schemas([Int]))  # one type is an OfAny still: where one type is meant, give it directly
    assert type(only) is S.OfAny.Value and only.schemas == (Int,) and store.Schemas.OfProperty(lambda p: p.schema([Int])).schema is Int
    assert (S.plain_forms(Float), S.plain_forms(Bytes), S.plain_forms(Labels), S.plain_forms(Tags), S.plain_forms(Grid)) == (
        {"float", "text"}, {"text"}, {"mapping"}, {"list"}, {"list", "mapping"})  # NaN is text, bytes base64, a tensor flat
    assert S.plain_forms(One) == {"int", "text", "mapping"} and S.plain_forms(Free) == {"mapping"}  # unrestricted: tagged
    assert S.plain_forms(S.OfScalar.Data(name="Word")) == {"bool", "int", "float", "text", "mapping", "list"}  # not a basic scalar
    assert S.OfAny.Value(schemas=(S.OfScalar.Data(),)).validate() == [
        "a schema: an unnamed scalar is not a basic scalar: a scalar is Schemas.OfBool, Schemas.OfInt, Schemas.OfFloat, Schemas.OfStr or Schemas.OfBytes"]
    assert store.Schemas.OfAny(lambda a: a.schemas([Tags, Grid])).validate() == [
        "an OfAny's schemas Tags and a container are not told apart on the wire"]
    assert store.Schemas.OfAny(lambda a: a.schemas([Cell, Labels])).validate() == [
        "an OfAny's schemas Cell and Labels are not told apart on the wire"]
    assert store.Schemas.OfAny(lambda a: a.schemas([store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("r").schema([Int])])),
                                                    Labels])).validate() == [
        "an OfAny's schemas a value object and Labels are not told apart on the wire"]  # an unnamed one: its kind
    assert One == One and One != Free  # schemas by identity
    Either = store.Schemas.OfUnion(lambda u: u.branches([
        lambda t: t.name("a").schema([Int]),
        lambda t: t.name("b").schema([Text]),
    ]))
    Both = store.Schemas.OfIntersection(lambda i: i.parts([
        lambda t: t.name("a").schema([Int]),
        lambda t: t.name("b").schema([Text]),
    ]))
    assert store.Schemas.OfAny(lambda a: a.schemas([Float, Text, Phone, Code, Labels, Either, Both, Tags])).validate() == [
        "an OfAny's schemas Schemas.OfFloat and Schemas.OfStr are not told apart on the wire",
        "an OfAny's schemas Phone and Code are not told apart on the wire",
        "an OfAny's schemas Phone and Labels are not told apart on the wire",
        "an OfAny's schemas Phone and a union are not told apart on the wire",
        "an OfAny's schemas Phone and an intersection are not told apart on the wire",
        "an OfAny's schemas Code and Labels are not told apart on the wire",
        "an OfAny's schemas Code and a union are not told apart on the wire",
        "an OfAny's schemas Code and an intersection are not told apart on the wire",
        "an OfAny's schemas Labels and a union are not told apart on the wire",
        "an OfAny's schemas Labels and an intersection are not told apart on the wire",
        "an OfAny's schemas a union and an intersection are not told apart on the wire"]
    with raises(TypeError, match="link 'schema' cannot link 3"):  # the builders link schemas, nothing else
        store.Schemas.OfAny(lambda a: a.schemas([3]))
    assert S.OfAny.Value((3, One, S.OfScalar.Data(int, description=True), Grid)).validate() == [  # data they refuse to build
        "an OfAny holds schemas, got 3", "an OfAny holds schemas of other kinds, not an OfAny",
        "Schemas.OfInt: a description is text, got bool"]
    Thing = store.Schemas.OfObject(lambda o: o.name("Thing").ref(True))
    assert store.Schemas.OfObject(lambda o: o.properties([
        lambda t: t.name("x").schema([store.Schemas.OfAny(lambda a: a.schemas([Int, Thing]))]),
    ])).validate() == ["property 'x': a reference object schema cannot be a property's schema"]  # a value only

## ANY-02 · Proxies: a restricted value written through the visitors as its schema is, inferred; an unrestricted one given as a value, built standalone where it is not a scalar

In [ ]:
with store.transaction():
    card = store.Card(lambda c: c.one(3).free("x"))
    assert (held(card.one), held(card.free)) == (3, "x") and card.one.get() == 3
    card = store.Card(lambda c: (
        c
        .one(lambda p: p.as_object(lambda o: o.number("1")))  # the one type of its kind
        .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a"))))
    ))
    assert (held(card.one), held(card.tags)) == ("Phone", "Tags") and card.one.get() is None
    with raises(TypeError, match="property 'one' does not hold a value of Tags"):
        store.Card(lambda c: c.one(store.Tags(lambda t: t)))
    with raises(TypeError, match="'Phone' is a value object schema; build a value through its owner, or standalone through "
                                "the store"):
        Proxies.OfObject.Builder(store, Phone, "Phone")
    with raises(ReadOnlyError, match="property 'one' does not hold a union value"):
        store.Card(lambda c: c.one(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.n(1)))))
    # Unrestricted: a scalar, or a value, standalone or another's; a Spec has no schema to build by.
    phone = store.Phone(lambda p: p.number("2"))
    tags = store.Tags(lambda t: t.at(0).as_scalar("t"))
    assert Proxies.owner_of(phone) is None and tags.as_schema() is Tags
    values = [phone, store.Code(lambda c: c.n(1)), store.Stamp(lambda s: s.at(1).by("me")), tags, card.one]
    assert [held(store.Card(lambda c: c.free(value)).free) for value in values] == ["Phone", "Code", "Stamp", "Tags", "Phone"]
    assert held(store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_object(card.one)))).free) == "Phone"
    fresh = store.Phone(lambda p: p.number("3"))
    first, second = store.Card(lambda c: c.free(fresh)), store.Card(lambda c: c.free(fresh))
    phones = []
    for each in (first, second):
        each.free.as_value(lambda a: a.as_object(lambda o: phones.append(o) or o))
    assert phones[0] is fresh and phones[1] is not fresh and Proxies.owner_of(fresh) is first  # adopted once, then copied
    unbuilt = "property 'free' holds a value of any schema: give a value, which carries its schema, not a Spec"
    with raises(ReadOnlyError, match=unbuilt):
        store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_object(lambda o: o.number("1")))))
    with raises(ReadOnlyError, match=unbuilt):
        store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.remove(0)))))
    read = []  # a Spec of another kind than the value held reads it as empty, and writes nothing
    store.Card(lambda c: c.one(3).one(lambda p: p.as_value(lambda a: a.as_object(lambda o: read.append(o.as_schema()) or o))))
    assert read == [None]
    found = []
    card.free.as_value(lambda a: a.as_scalar(lambda s: found.append((s.as_schema(), s.get())) or s))
    assert found == [(Free, None)]  # nothing held: a scalar of any schema, absent
    listed = store.Card(lambda c: c.many(lambda p: p.as_value(lambda a: a.as_container(
        lambda c: c.at(0).as_scalar(1) and c.at(1).as_object(phone)))))
    elements = []
    listed.many.as_value(lambda a: a.as_container(lambda c: c.as_entries(elements.extend)))
    assert [index for index, _ in elements] == [0, 1]
    with raises(TypeError, match="property 'one' does not hold a value of a container"):
        store.Card(lambda c: c.one(listed.many))

## ANY-03 · Snapshots: a restricted value bare, an unrestricted one tagged, read back; malformed ones refused, with a path

In [ ]:
other = B.Store()  # another store of the same schemas
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own


def many(c):
    c.at(0).as_scalar(2.5)
    c.at(1).as_object(phone)
    c.at(3).as_container(tags)  # nothing at 2
    return c


with store.transaction():
    card = store.Card(lambda c: (
        c
        .one(lambda p: p.as_object(lambda o: o.number("1")))
        .free(store.Code(lambda x: x.s("c")))
        .many(lambda p: p.as_value(lambda a: a.as_container(many)))
        .tags(True)
    ))
written = root_of(Plain.ToPlain(store).OfObject(Card, card))
assert written == {"one": {"number": "1"}, "free": {"Code": {"s": "c"}}, "tags": True,
                   "many": [{"float": 2.5}, {"Phone": {"number": "2"}}, None, {"Tags": ["t"]}]}
for encode, decode in [(JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    text_ = encode(store).OfObject(Card, card)
    assert encode(other).OfObject(other.Card, decode(other).OfObject(other.Card, text_)) == text_
with store.transaction():
    assert root_of(Plain.ToPlain(store).OfObject(Card, store.Card(lambda c: c.one("s").free(b"\x01")))) == {
        "one": "s", "free": {"bytes": "AQ=="}}
assert Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) is not card  # counted: made anew
back = Plain.FromPlain(other).OfObject(other.Card, document(single("Card", {"one": 3, "tags": ["a"]})))
assert (held(back.one), held(back.tags)) == (3, "Tags")  # each read as the one type its plain form fits
assert held(Plain.FromPlain(other).OfObject(other.Card, document(single("Card", {"one": "s"}))).one) == "s"


def refused(values, message):
    at = '$.mutations[0].upsert.exists[0].spec[0]["0"][0].as_property.mutations[0].set.value'  # its one member's value
    with raises(DecodeError, match=at + message):
        Plain.FromPlain(other).OfObject(other.Card, document(single("Card", values)))


refused({"one": True}, ": a value of any schema takes the plain form of one of its schemas, got bool")
refused({"one": 2.5}, ": a value of any schema takes the plain form of one of its schemas, got float")
refused({"one": None}, ": a value of any schema takes the plain form of one of its schemas, got NoneType")
refused({"free": 3}, ": a value of any schema is {schema: value}, its schema a basic scalar's tag or a registered "
                     "schema's name")
refused({"free": {"Nope": 1}}, ".Nope: no schema registered as 'Nope'")
refused({"free": {"Card": {}}}, ".Card: 'Card' is not a value's schema")
refused({"free": {"int": "x"}}, ".int: expected int, got str")
with store.transaction():
    with raises(TypeError, match="a value of any schema holds one of its schemas; float is not one"):
        Plain.ToPlain(store).OfObject(Card, store.Card(lambda c: c.one(2.5)))  # builders do not check scalars' types
with store.transaction():
    Point = store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("x").schema([Int])]))  # unnamed: not registered
    Spot = store.Schemas.OfObject(lambda o: o.name("Spot").ref(True).properties([lambda t: t.name("at").schema([Point])]))
    loose = store.Card(lambda c: c.free(store.Spot(lambda s: s.at(lambda p: p.as_object(lambda o: o.x(1)))).at))
    with raises(TypeError, match="a value of any schema is of a registered schema; None is not registered"):
        Plain.ToPlain(store).OfObject(Card, loose)

## ANY-04 · Validation and comparison under the schema a value holds; uniqueness by value

In [ ]:
with store.transaction():
    validate = Validators.Validate(store)
    assert validate(Card, card) == []
    assert validate(Card, store.Card(lambda c: c.one(2.5).many(lambda p: p.as_value(lambda a: a.as_container(
        lambda c: c.at(0).as_object(phone)))))) == ["Card#0.one: float is not one of its schemas"]
    assert validate(Card, loose) == ["Card#0.free: a value of any schema is of a registered schema; None is not registered"]
    assert validate(Card, store.Card(lambda c: c.free(store.Code(lambda x: x.n("x"))))) == ["Card#0.free.n: expected int, got str"]


    def compare(a, b):
        return C.OfObject(Card, a).compare(C.OfObject(Card, b))


    assert compare(store.Card(lambda c: c.one(3).free(phone)), store.Card(lambda c: c.one(3).free(store.Phone(lambda p: p.number("2"))))) == 0
    assert compare(store.Card(lambda c: c.one(3)), store.Card(lambda c: c.one("3"))) is None  # of different types: incomparable
    assert compare(store.Card(lambda c: c.free(phone)), store.Card(lambda c: c.free(store.Code(lambda x: x.s("2"))))) is None


    def holding(s, free):
        """Holds, whose entries hold a value of any schema (`free`), and Holder, built in the store `s`."""
        holds = s.Schemas.OfRelation(lambda r: (
            r
            .name("Holds")
            .links([
                lambda t: t.name("card"),
                lambda t: t.name("other"),
            ])
            .properties([
                lambda t: t.name("what").schema([free]),
                lambda t: t.name("note").schema([s.Schemas.OfStr]),
            ])
            .uniques("note")
        ))
        return holds, s.Schemas.OfObject(lambda o: (
            o
            .name("Holder")
            .ref(True)
            .adjacencies([
                lambda t: t.name("holds").relation([holds]).me("card"),
                lambda t: t.name("held").relation([holds]).me("other"),
            ])
        ))


    Holds, Holder = holding(store, Free)
    target = store.Holder(lambda h: h)
    holder = store.Holder(lambda h: h.holds(lambda a: (
        a
        .insert(lambda e: e.other(target).what(phone).note("a"))
        .insert(lambda e: e.other(target).what(store.Phone(lambda p: p.number("2"))).note("b"))
        .insert(lambda e: e.other(target).what(1).note("c"))
        .insert(lambda e: e.other(target).note("d"))
    )))
    assert validate(Holder, holder) == [
        "unique(note) violated: entries agreeing on ['card', 'other', 'what'] differ on ['note']"]  # equal values: one key
with other.transaction():
    holding(other, other.Card.as_schema().properties["free"].schema)  # built in other too, its own
snapshot = Plain.ToPlain(store).Reachable(Holder, holder)
assert [e.get("what") for e in root_of(snapshot)["holds"]] == [{"Phone": {"number": "2"}}] * 2 + [{"int": 1}, None]
assert Plain.ToPlain(other).Reachable(other.Holder, Plain.FromPlain(other).Reachable(other.Holder, snapshot)) == snapshot
again = Plain.FromPlain(store).Reachable(Holder, snapshot)  # counted: made anew, with its entries
assert again is not holder and len(again.holds) == 4 and len(holder.holds) == 4
read = Plain.FromPlain(other).OfObject(other.Card, Plain.ToPlain(store).OfObject(Card, card))  # other's own Card
OtherCard = other.Card.as_schema()
with other.transaction():
    assert C.OfObject(OtherCard, read).compare(C.OfObject(OtherCard, other.Card(read))) == 0  # a container of values of any
    # type, element by element: an object and its clone

## ANY-05 · Snapshots write a value of any schema's schemas, linked, and read them back

In [ ]:
typed = B.Store()  # a snapshot holds every schema of its store: these, typed by the basic scalars
with typed.transaction():
    Phoned = typed.Schemas.OfObject(lambda o: o.name("Phone").properties([lambda t: t.name("number").schema([typed.Schemas.OfStr.as_schema()])]))
    typed.Schemas.OfObject(lambda o: (
        o
        .name("Card")
        .ref(True)
        .properties([
            lambda t: t.name("one").schema([typed.Schemas.OfAny(lambda a: a.schemas([typed.Schemas.OfInt.as_schema(), typed.Schemas.OfStr.as_schema(), Phoned]))]),
            lambda t: t.name("free").schema([typed.Schemas.OfAny(lambda a: a)]),
        ])
    ))
log = Plain.ToPlain(typed).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log  # read back, written again: the same
one, free = (back.registered("Card").properties[name].schema for name in ("one", "free"))
assert type(one) is S.OfAny.Value and one.schemas[2] is back.registered("Phone") and one.schemas[0] is back.Schemas.OfInt.as_schema()
assert free.schemas == ()